In [76]:
include("CRD_STA.jl")
include("ALST_BEK.jl")
using NonlinearEigenproblems
using LinearAlgebra
using Plots

In [67]:
function BF(N_cheb,Ro,Tw,Mr)
    gamma = 1.4
    sigma = 0.72
    Co = 2 - Ro - Ro^2
    u0,v0,w0,f,q,D,D2,x = baseflow_var(N_cheb,Ro,Co)
    H,T = T_ca(Mr,f,q,w0,gamma,Tw)
    F,G,H,T,rho,z = interp(u0,v0,H,T,x,N_cheb,"sim")
    lam = - (2/3) * T
    kappa = (1/sigma) * T
    return F,G,H,T,rho,x,lam,kappa,D,D2
end
function eigsol(F,G,H,rho,lam,kappa,T,sigma,gamma,R,Ma,omega,be,N_cheb,Ro,Co,D,D2,c,num)
    cof = Spatial_mode_BEK(F,G,H,rho,lam,kappa,T,sigma,gamma,R,Ma,N_cheb,Ro,Co,D,D2)
    L0,L1,L2 = assemble_mat(cof,D,D2,be,omega)
    nep = PEP([L0,L1,L2]); 
    eigval,eigvec = iar(nep,σ = c , neigs = num ,maxit = 500,tol=1e-12)
    # eigval = conj(eigval)
    return cof,eigval,eigvec
end
function integral(x,y)
    inte_progress =  zeros(ComplexF64,length(x),1)
    for i = 1 : length(x)
        inte_progress[i] = x[1:i,1]' * y[1:i,1]
    end
    return inte_progress
end
function mat_diff(A,B,N_cheb)
    block_size = N_cheb + 1
    nblock = 5
    C = zeros(ComplexF64,block_size*nblock,block_size*nblock)
    for i = 0 : nblock - 1
        for j = 0 : nblock - 1
        C[1+block_size*i:block_size*(i+1),1+block_size*j:block_size*(j+1)] = (B * A[1+block_size*i:block_size*(i+1),1+block_size*j:block_size*(j+1)])
        end
    end

    return C'
end
function eig_full(eigvec,N_cheb,num)
    N = N_cheb + 1
    eigvec = eigvec[:,num]
    insert!(eigvec,5N-8,0im)
    insert!(eigvec,4N-6,0im)
    insert!(eigvec,4N-6,0im)
    insert!(eigvec,3N-4,0im)
    insert!(eigvec,3N-4,0im)
    insert!(eigvec,2N-2,0im)
    insert!(eigvec,2N-2,0im)
    insert!(eigvec,N,0im)
    insert!(eigvec,N,0im)
    rho = eigvec[1:N]
    u = eigvec[N+1:2N]
    v = eigvec[2N+1:3N]
    w = eigvec[3N+1:4N]
    T = eigvec[4N+1:5N]
    return eigvec,(rho,u,v,w,T)
end

eig_full (generic function with 1 method)

In [68]:
N_cheb = 199
Ro = -1
Tw = 1.0
Mr = 0.3
gamma = 1.4
sigma = 0.72
Co = 1
F,G,H,T,rho,z,lam,kappa,D,D2 = BF(N_cheb,Ro,Tw,Mr)

([0.0; 0.0001398472338365777; … ; 6.744959925293321e-24; 6.744959925293321e-24;;], [0.0; -0.00016886050527051708; … ; -1.0; -1.0;;], [-3.176003061707337e-25; -3.8350176579340704e-8; … ; -0.884473922139928; -0.884473922139928;;], [1.0; 1.0000023059849867; … ; 0.9999997873656139; 0.9999997873656139;;], [1.0; 0.9999976940203308; … ; 1.0000002126344314; 1.0000002126344314;;], [0.0; 0.00027415889944683725; … ; 40.0; 40.0;;], [-0.6666666666666666; -0.6666682039899912; … ; -0.6666665249104092; -0.6666665249104092;;], [1.3888888888888888; 1.388892091645815; … ; 1.3888885935633526; 1.3888885935633526;;], [-6000.227272728362 7295.4609596231385 … -0.4545737768742528 0.22727272727272727; -1823.6545211491878 911.7704454173754 … 0.22727479032807618 -0.11363031451756492; … ; 1.3587647399973356e-9 -2.717698817425532e-9 … -1.0902737860619867e-5 2.1806834485978705e-5; -0.0 0.0 … -0.0 0.0], [2.1603598651886173e7 -3.4328129413350314e7 … 5446.344916291144 -2723.0014267909523; 9.97813376224312e6 -1.42678804

In [74]:
n = 30
num = 1
R = 400
omega = 7.9/R
be = n/R
eye = I(N_cheb + 1)
Zeros = zeros(N_cheb + 1, N_cheb + 1);
A0_11 = im * (be * G .- omega) .* eye - (1/R) * (D2 - be^2 .* eye - F .* eye + H.*D )
A1_11 = im * F .* eye
A2_11 = (1/R) * eye
A0_12 = -(1/R) * (-2 * (G.+1)).* eye
A1_12 = A2_12 = Zeros
A0_13 = A1_13 = A2_13 = Zeros
A0_14 = A2_14 = Zeros
A1_14 = im * eye

A0_21 = -(1/R) * (2 * (G .+ 1)) .* eye
A1_21 = A2_21 = Zeros
A0_22 = im * (be * G .- omega) .* eye - (1/R) * (D2 - be^2 .* eye - F .* eye + H.*D )
A1_22 = im * F .* eye
A2_22 = (1/R) * eye
A0_23 = A1_23 = A2_23 = Zeros
A0_24 = im * be .* eye
A1_24 = A2_24 = Zeros

A0_31 = D * F .* eye
A1_31 = A2_31 = Zeros
A0_32 = D * G .* eye 
A1_32 = A2_32 = Zeros
A0_33 = im * (be .-omega) .* eye - (1/R) * (D2 - be^2 .* eye + H.*D - D*H.*eye)
A1_33 = im * F .* eye
A2_33 = (1/R) * eye
A0_34 = D 
A1_34 = A2_34 = Zeros

A0_41 = im * (im/R) .* eye
A1_41 = im * eye
A2_41 = Zeros
A0_42 = im * be .* eye
A1_42 = A2_42 = Zeros
A0_43 = D
A1_43 = A2_43 = Zeros 
A0_44 = A1_44 = A2_44 = Zeros

A0 = [A0_11 A0_12 A0_13 A0_14;A0_21 A0_22 A0_23 A0_24;A0_31 A0_32 A0_33 A0_34;A0_41 A0_42 A0_43 A0_44]
A1 = [A1_11 A1_12 A1_13 A1_14;A1_21 A1_22 A1_23 A1_24;A1_31 A1_32 A1_33 A1_34;A1_41 A1_42 A1_43 A1_44]
A2 = [A2_11 A2_12 A2_13 A2_14;A2_21 A2_22 A2_23 A2_24;A2_31 A2_32 A2_33 A2_34;A2_41 A2_42 A2_43 A2_44]

A0 = A0[setdiff(1:end , (1,N_cheb + 1,N_cheb + 2,2N_cheb + 2,2N_cheb + 3,3N_cheb + 3,4N_cheb + 4)),setdiff(1:end , (1,N_cheb + 1,N_cheb + 2,2N_cheb + 2,2N_cheb + 3,3N_cheb + 3,4N_cheb + 4))]
A1 = A1[setdiff(1:end , (1,N_cheb + 1,N_cheb + 2,2N_cheb + 2,2N_cheb + 3,3N_cheb + 3,4N_cheb + 4)),setdiff(1:end , (1,N_cheb + 1,N_cheb + 2,2N_cheb + 2,2N_cheb + 3,3N_cheb + 3,4N_cheb + 4))]
A2 = A2[setdiff(1:end , (1,N_cheb + 1,N_cheb + 2,2N_cheb + 2,2N_cheb + 3,3N_cheb + 3,4N_cheb + 4)),setdiff(1:end , (1,N_cheb + 1,N_cheb + 2,2N_cheb + 2,2N_cheb + 3,3N_cheb + 3,4N_cheb + 4))]


793×793 Matrix{Float64}:
 0.0025  0.0     0.0     0.0     0.0     …  0.0  0.0  0.0  0.0  0.0  0.0  0.0
 0.0     0.0025  0.0     0.0     0.0        0.0  0.0  0.0  0.0  0.0  0.0  0.0
 0.0     0.0     0.0025  0.0     0.0        0.0  0.0  0.0  0.0  0.0  0.0  0.0
 0.0     0.0     0.0     0.0025  0.0        0.0  0.0  0.0  0.0  0.0  0.0  0.0
 0.0     0.0     0.0     0.0     0.0025     0.0  0.0  0.0  0.0  0.0  0.0  0.0
 0.0     0.0     0.0     0.0     0.0     …  0.0  0.0  0.0  0.0  0.0  0.0  0.0
 0.0     0.0     0.0     0.0     0.0        0.0  0.0  0.0  0.0  0.0  0.0  0.0
 0.0     0.0     0.0     0.0     0.0        0.0  0.0  0.0  0.0  0.0  0.0  0.0
 0.0     0.0     0.0     0.0     0.0        0.0  0.0  0.0  0.0  0.0  0.0  0.0
 0.0     0.0     0.0     0.0     0.0        0.0  0.0  0.0  0.0  0.0  0.0  0.0
 ⋮                                       ⋱                      ⋮         
 0.0     0.0     0.0     0.0     0.0        0.0  0.0  0.0  0.0  0.0  0.0  0.0
 0.0     0.0     0.0     0.0     0.0     …

In [75]:
nep = PEP([A0,A1,A2]);
eigval,eigvec = iar(nep, σ = 0.6, neigs = 3,maxit = 500 , tol=1e-14)
# vel_full,vel = eig_full(eigvec,N_cheb,1)
eigval

3-element Vector{ComplexF64}:
 0.6122300847210326 + 0.05497693618625815im
 0.5446285443454266 + 0.04027346262377594im
 0.6865038899088812 + 0.06687901508168037im